# Explore dataset

**Auth**
- **Local:** put `KAGGLE_API_TOKEN=...` in a `.env` file next to this notebook (project folder).
- **Colab:** you do **not** get your project `.env`. Use the key icon (Secrets) → add `KAGGLE_API_TOKEN` → enable notebook access.

In [ ]:
# install once on Colab (skip locally if already installed)
import sys
if "google.colab" in sys.modules:
    %pip install -q kagglehub python-dotenv pillow matplotlib

In [ ]:
import os
from pathlib import Path
import json
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageOps
import kagglehub

# Colab Secrets first, else local .env
try:
    from google.colab import userdata
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    print("Loaded token from Colab Secrets")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()  # reads .env in the project folder
    print("Loaded token from .env")

assert os.getenv("KAGGLE_API_TOKEN"), "Missing KAGGLE_API_TOKEN (.env locally, or Colab Secrets)"

DATA = Path(kagglehub.competition_download("traffic-sign-object-detection-challenge"))
print(DATA)

In [ ]:
with open(DATA / "train" / "annotations.json", encoding="utf-8") as f:
    train = json.load(f)

classes = {c["id"]: c["name"] for c in train["categories"]}
print("classes:", classes)
print("images:", len(train["images"]), "boxes:", len(train["annotations"]))

counts = Counter(a["category_id"] for a in train["annotations"])
for i, n in sorted(counts.items()):
    print(f"  {classes[i]}: {n}")

In [ ]:
info = train["images"][0]
# phone photos often have EXIF rotation — apply it so boxes match the labels
img = ImageOps.exif_transpose(Image.open(DATA / "train" / "images" / info["file_name"])).convert("RGB")
print("json size:", info["width"], info["height"], "| image size:", img.size)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img)

for a in train["annotations"]:
    if a["image_id"] != info["id"]:
        continue
    x, y, w, h = a["bbox"]
    ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor="lime", linewidth=2))
    ax.text(x, y - 5, classes[a["category_id"]], color="yellow", fontsize=8)

ax.set_title(info["file_name"])
ax.axis("off")
plt.show()